[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilay47/marginally-realistic/blob/main/cvpr27/notebooks/nb4v2_eval.ipynb)

# nb4v2: evaluation of the v2 runs
Same as nb4 (E2-E8, 3 draws x 1,024, 250-step ancestral, subsampling CIs for FVD/MMD) but for `joint_v2`, `video_only_v2`, `sensor_only_v2` at T=32 (`--suffix v2`; outputs in `eval_v2/`, samples in `eval_samples/v2/`). Needs the three v2 runs finished (`log.json`): run each notebook's finalize cell if you stopped early.

## 1. GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "nvidia-smi: no output")
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime -> Change runtime type -> A100 GPU, then rerun.")
print("cuda:", torch.cuda.get_device_name(0), "| mem GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

## 2. Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
ROOT = "/content/drive/MyDrive/physgen_cvpr27"
os.makedirs(ROOT, exist_ok=True)

## 3. Code

In [ ]:
import os
if os.path.exists("/content/repo"):
    !git -C /content/repo pull
else:
    !git clone https://github.com/nilay47/marginally-realistic.git /content/repo
%cd /content/repo
!git log -1 --format="commit %H%n%s"

## 4. Install

In [ ]:
# torch / torchvision come with Colab. pytorchvideo is the second I3D source for FVD (see videofeat.py).
!pip install -q scipy matplotlib pytorchvideo

## 5. T=32 environment

In [ ]:
# v2 runs use T=32 frames. These must be set BEFORE cvpr27 is imported (physvid reads them at import).
%env PV_T=32
%env PV_DT=0.04
%env PV_WIN=8

## 6. Check inputs

In [ ]:
import os
for r in ["joint_v2", "video_only_v2", "sensor_only_v2"]:
    assert os.path.exists(f"{ROOT}/{r}/log.json"), f"{r} not finished (no log.json): finish or finalize it first"
for f in ["cache/pendulum_T32_dt0.04_heldout.npz", "cache/pendulum_T32_dt0.04_train.npz", "cache/video_metric.json"]:
    assert os.path.exists(f"{ROOT}/{f}"), f"missing {f}"
print("inputs present")

## 7. Sample + evaluate

In [ ]:
%%bash
set -o pipefail
export PV_T=32 PV_DT=0.04 PV_WIN=8
mkdir -p /content/drive/MyDrive/physgen_cvpr27/eval_v2_logs
cd /content/repo
PYTHONUNBUFFERED=1 python -u -m cvpr27.eval --suffix v2 2>&1 | tee -a "/content/drive/MyDrive/physgen_cvpr27/eval_v2_logs/eval_$(date +%s).log"

## 8. Figures (PDF + PNG)

In [ ]:
%%bash
export PV_T=32 PV_DT=0.04 PV_WIN=8
cd /content/repo && python -u -m cvpr27.figures --suffix v2

## 9. Summary

In [ ]:
from IPython.display import Markdown, Image, display
import glob
display(Markdown(open(f"{ROOT}/eval_v2/results.md").read()))
for p in sorted(glob.glob(f"{ROOT}/eval_v2/figures/*.png")):
    print(p); display(Image(p))